# Simple Linear Regression — OLS Foundations & Diagnostics — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 3

---
**Learning Objectives:**
- Distinguish regression from correlation — and recognise that neither implies causation
- Derive the OLS estimators $\hat{\beta}_0$ and $\hat{\beta}_1$ from the least-squares principle
- Reproduce the worked example by hand and verify it with `statsmodels`
- Compute fitted values $\hat{Y}$ and residuals $\hat{u}$, and check the two mechanical properties $\sum \hat{u} = 0$ and $\sum x \hat{u} = 0$
- Spot violations of CLRM assumptions (heteroskedasticity, autocorrelation, outliers) visually in residual plots
- Apply OLS to a real-world hedge-ratio estimation in Python

> Run each cell with **Shift+Enter**. This notebook accompanies the lecture slides.


## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'
print('✓ Libraries loaded.')

---
# Part 1 — From Correlation to Regression

Before any math, let's nail down the difference between **correlation** and **regression**.

| | Correlation | Regression |
|--|--|--|
| **Symmetric?** | Yes — corr(x,y) = corr(y,x) | No — y on x ≠ x on y |
| **Direction** | None | You pick: y is dependent, x is independent |
| **Output** | Single number ∈ [−1, 1] | Slope, intercept, fitted line |
| **Implies causation?** | **No** | **Also no** |

### 1.1 Demonstration: same data, swap x and y

In [ ]:
np.random.seed(42)
x = np.random.normal(0, 2, 80)
y = 1.2 + 0.7 * x + np.random.normal(0, 1.5, 80)

# Correlation is the same either way
corr_xy = np.corrcoef(x, y)[0, 1]
corr_yx = np.corrcoef(y, x)[0, 1]
print(f'corr(x, y) = {corr_xy:.4f}')
print(f'corr(y, x) = {corr_yx:.4f}   ← identical')

# Regression slopes differ
slope_y_on_x, intercept_y_on_x = np.polyfit(x, y, 1)   # y = a + b·x
slope_x_on_y, intercept_x_on_y = np.polyfit(y, x, 1)   # x = a + b·y
print(f'\nRegression y on x: slope β1_hat = {slope_y_on_x:.4f}')
print(f'Regression x on y: slope β1_hat = {slope_x_on_y:.4f}   ← different!')
print('→ Regression IS directional. The slope depends on which variable you put on the left.')

In [ ]:
# Visualise: two regression lines on the same scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax in axes:
    ax.scatter(x, y, color=GREY, alpha=0.6, s=40, edgecolor='black', linewidth=0.4)
    ax.axhline(0, color=GREY, lw=0.5); ax.axvline(0, color=GREY, lw=0.5)
    ax.set_xlabel('x'); ax.set_ylabel('y')

xx = np.linspace(x.min(), x.max(), 100)
axes[0].plot(xx, slope_y_on_x * xx + intercept_y_on_x, color=RED, lw=2)
axes[0].set_title(f'Regress y on x:  β1_hat = {slope_y_on_x:.3f}', fontweight='bold')

# For x on y, line is x = a + b·y → reorganise to y = (x − a) / b
yy = np.linspace(y.min(), y.max(), 100)
axes[1].plot(slope_x_on_y * yy + intercept_x_on_y, yy, color=BLUE, lw=2)
axes[1].set_title(f'Regress x on y:  β1_hat = {slope_x_on_y:.3f}', fontweight='bold')

plt.suptitle('Same scatter, two different regression lines — regression is asymmetric',
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### 1.2 The causality warning
Regression describes a *conditional mean*: "on average, y given x is ...". That is **not** a causal claim. The classic example: a regression of drowning deaths on ice cream sales gives a strongly positive significant $\hat{\beta}_1$ — but ice cream does not cause drownings. **Both** are driven by hot weather.

**Rule for your career:** whenever you see regression output, ask *"is this a description or a causal claim?"* The math doesn't distinguish — you have to.

---
# Part 2 — The OLS Idea

The simple linear regression model is
$$y_t = \beta_0 + \beta_1 x_t + u_t$$
where β₀ and β₁ are unknown true parameters and u is the error term.

**OLS principle:** choose $\hat{\beta}_0$ and $\hat{\beta}_1$ that **minimise the sum of squared residuals** $\sum \hat{u}$².

Why squared (not absolute)? Three reasons: (i) positive and negative residuals can't cancel out, (ii) large errors are penalised more heavily, (iii) the math is differentiable so we get a closed-form solution.

In [ ]:
# Visualise residuals as vertical distances
np.random.seed(7)
x_demo = np.linspace(0, 7, 8)
y_demo = 0.8 + 0.9 * x_demo + np.random.normal(0, 0.7, 8)
slope_d, intercept_d = np.polyfit(x_demo, y_demo, 1)
yhat_d = slope_d * x_demo + intercept_d

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x_demo, y_demo, color='black', s=80, zorder=5, label='Data')
xline = np.linspace(x_demo.min()-0.3, x_demo.max()+0.3, 100)
ax.plot(xline, slope_d * xline + intercept_d, color=RED, lw=2, label='OLS line')
for xi, yi, yhi in zip(x_demo, y_demo, yhat_d):
    ax.plot([xi, xi], [yi, yhi], color=ORANGE, lw=2)
ax.set_title('OLS picks the line that minimises the sum of squared orange segments',
             fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(loc='upper left', frameon=False)
plt.tight_layout(); plt.show()

print(f'Sum of squared residuals: {((y_demo - yhat_d)**2).sum():.4f}')
print('Any other line through these points has a LARGER sum.')

---
# Part 3 — Worked Example by Hand: Hedge-Ratio Mini-Case

Same 5-day data as the lecture slides. Goal: estimate $\hat{\beta}_0$ and $\hat{\beta}_1$ **without any library** — just `numpy` arithmetic.

**OLS formulas (Brooks, eq. 3.10–3.11):**
$$\hat{\beta}_1 = \frac{\sum (x - \bar{x})(y - \bar{y})}{\sum (x - \bar{x})^2} = \frac{\text{Cov}(x, y)}{\text{Var}(x)}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \, \bar{x}$$

In [ ]:
# The dataset from the slides — 5 trading days
x = np.array([-2, -1,  0,  1,  2], dtype=float)   # Δ Futures (%)
y = np.array([-1.5, -0.5, 0,  1,  2.0], dtype=float) # Δ Spot (%)

print('Day  | x (Δ Futures)  | y (Δ Spot)')
print('-' * 40)
for i, (xi, yi) in enumerate(zip(x, y), start=1):
    print(f' {i}   |   {xi:+.1f}          |   {yi:+.1f}')

In [ ]:
# Step 1 — Compute the means
x_bar = x.mean()
y_bar = y.mean()
print(f'x_bar = {x_bar}')
print(f'y_bar = {y_bar}')

In [ ]:
# Step 2 — Sum of products of deviations  (numerator of β1_hat)
num = ((x - x_bar) * (y - y_bar)).sum()
print(f'∑ (x − x_bar)(y − y_bar) = {num}')

# Step 3 — Sum of squared deviations of x  (denominator of β1_hat)
den = ((x - x_bar) ** 2).sum()
print(f'∑ (x − x_bar)²       = {den}')

In [ ]:
# Step 4 — Slope estimator β1_hat
beta1_hat = num / den
print(f'β1_hat = {num} / {den} = {beta1_hat}')

# Step 5 — Intercept estimator β0_hat
beta0_hat = y_bar - beta1_hat * x_bar
print(f'β0_hat = y_bar − β1_hat · x_bar = {y_bar} − {beta1_hat} · {x_bar} = {beta0_hat}')

print(f'\nResult:  y_hat = {beta0_hat:.2f} + {beta1_hat:.2f} · x')
print(f'Interpretation: the hedge ratio is {beta1_hat:.2f} — for every 1% move in futures,')
print(f'                spot is expected to move {beta1_hat:.2f}% in the same direction.')

### 3.1 Geometric check — the OLS line always passes through ($\bar{x}$, $\bar{y}$)

In [ ]:
y_at_xbar = beta0_hat + beta1_hat * x_bar
print(f'OLS prediction at x = x_bar = {x_bar}:  y_hat = {y_at_xbar}')
print(f'Mean of y:                         y_bar = {y_bar}')
print(f'Equal? {np.isclose(y_at_xbar, y_bar)}   ← always true by construction')

---
# Part 4 — The Same Thing with `statsmodels`

Now let's verify the hand calculation using the standard Python OLS library.

Critical step: `sm.add_constant(X)` prepends a column of ones so the model estimates an intercept $\hat{\beta}_0$. **Forget this and your regression has no intercept** — the line is forced through the origin, which is almost never what you want.

In [ ]:
X = sm.add_constant(x)            # adds intercept column
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# Just the coefficient column — the rest comes in V5
print('coef from statsmodels:')
print(f'  const  (β0_hat) = {model.params[0]:.4f}')
print(f'  x1     (β1_hat) = {model.params[1]:.4f}')
print(f'\ncoef from our hand calculation:')
print(f'  β0_hat = {beta0_hat:.4f}')
print(f'  β1_hat = {beta1_hat:.4f}')
print(f'\nIdentical? {np.allclose(model.params, [beta0_hat, beta1_hat])} ✓')

---
# Part 5 — Fitted Values & Residuals in Action

Once we have $\hat{\beta}_0$ and $\hat{\beta}_1$, two derived quantities matter:
- **Fitted value:** $\hat{Y}_t = \hat{\beta}_0 + \hat{\beta}_1 x_t$ — what the model predicts
- **Residual:** $\hat{u}_t = y_t - \hat{Y}_t$ — what the model misses


In [ ]:
yhat = beta0_hat + beta1_hat * x
uhat = y - yhat
uhat_sq = uhat ** 2

tbl = pd.DataFrame({
    't': range(1, 6),
    'x': x, 'y': y,
    'Y_hat': yhat.round(4),
    'u_hat = y − Y_hat': uhat.round(4),
    'u_hat²': uhat_sq.round(6),
}).set_index('t')

print(tbl)
print('-' * 50)
print(f'Σ x          = {x.sum():.4f}')
print(f'Σ y          = {y.sum():.4f}')
print(f'Σ Y_hat          = {yhat.sum():.4f}')
print(f'Σ u_hat          = {uhat.sum():.4f}    ← exactly zero')
print(f'Σ x · u_hat      = {(x*uhat).sum():.4f}    ← exactly zero')
print(f'Σ u_hat² (SSR)   = {uhat_sq.sum():.4f}')

### 5.1 The two mechanical properties — these are NOT assumptions

$\sum \hat{u} = 0$ and $\sum x \hat{u} = 0$ are not assumptions about the world. They are **consequences of the OLS minimisation** with an intercept:
- The intercept $\hat{\beta}_0$ is precisely the constant that forces $\sum \hat{u} = 0$
- The slope $\hat{\beta}_1$ is precisely the slope that forces $\sum x \hat{u} = 0$

They will always hold *in your sample*, regardless of whether the model is true.

In [ ]:
# Visualise the fit and the residuals
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.scatter(x, y, color='black', s=110, zorder=5, label='Observation (y)')
xline = np.linspace(x.min()-0.3, x.max()+0.3, 50)
ax.plot(xline, beta0_hat + beta1_hat * xline, color=RED, lw=2, label='OLS line')
ax.scatter(x, yhat, marker='s', color=RED, s=70, zorder=5, label='Fitted Y_hat')
for xi, yi, yh in zip(x, y, yhat):
    ax.plot([xi, xi], [yi, yh], color=ORANGE, lw=2)
ax.set_xlabel('Δ Futures (x, %)'); ax.set_ylabel('Δ Spot (y, %)')
ax.set_title('Fit and residuals', fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.axhline(0, color=GREY, lw=0.5); ax.axvline(0, color=GREY, lw=0.5)

ax = axes[1]
ax.stem(x, uhat, basefmt=' ', linefmt=GREY)
ax.scatter(x, uhat, color=ORANGE, s=80, zorder=5, edgecolor='black', linewidth=0.6)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Δ Futures (x, %)'); ax.set_ylabel('Residual u_hat')
ax.set_title(f'Residuals — Σu_hat = {uhat.sum():.2f} (exactly zero)',
             fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

---
# Part 6 — Real-World Application: Gold–Silver Hedge Ratio

Now let's run OLS on real market data. We'll estimate the hedge ratio between gold futures (GC=F) and silver futures (SI=F) — two precious metals that tend to move together.

Setup: regress **gold returns** on **silver returns**. $\hat{\beta}_1$ tells us how much gold moves for every 1% silver move.

In [ ]:
# Download both futures series
px = yf.download(['GC=F', 'SI=F'], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']

# Daily percentage returns
ret = px.pct_change().dropna() * 100
print(f'Observations: {len(ret)} trading days')
print(f'\nDescriptive stats (daily returns in %):')
print(ret.describe().round(3))

In [ ]:
y = ret['GC=F']                       # gold returns
X = sm.add_constant(ret['SI=F'])       # silver returns + constant

model = sm.OLS(y, X).fit()

# Today: only the coef column. (std err, t, p-value, R² → V5)
print(f'Intercept β0_hat = {model.params[0]:.4f}  (daily % when silver is flat)')
print(f'Slope    β1_hat = {model.params[1]:.4f}  (gold-silver hedge ratio)')
print(f'\nFor every 1% move in silver, gold moves about {model.params[1]:.2f}% on average.')

### 6.1 Always plot diagnostic charts
The summary table tells you the numbers. The diagnostic plots tell you *whether to trust them*. Two charts you should look at on every regression:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: fitted line on the scatter
ax = axes[0]
ax.scatter(ret['SI=F'], y, color=GREY, alpha=0.5, s=15, edgecolor='black', linewidth=0.2)
xx = np.linspace(ret['SI=F'].min(), ret['SI=F'].max(), 100)
ax.plot(xx, model.params[0] + model.params[1] * xx, color=RED, lw=2,
        label=f'OLS:  y_hat = {model.params[0]:.3f} + {model.params[1]:.3f}·x')
ax.set_xlabel('Silver return (%)'); ax.set_ylabel('Gold return (%)')
ax.set_title('Fitted line', fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=False)

# Right: residual plot
ax = axes[1]
ax.scatter(model.fittedvalues, model.resid, color=GREY, alpha=0.5, s=15,
           edgecolor='black', linewidth=0.2)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Fitted Y_hat'); ax.set_ylabel('Residual u_hat')
ax.set_title('Residual plot — look for fanning or patterns',
             fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

---
# Part 7 — Spotting CLRM Violations in Residual Plots

Recall the four classical assumptions: A1 E[u]=0, A2 Var(u)=σ², A3 Cov(uₜ,uₛ)=0, A4 Cov(u,x)=0. When they hold, Gauss-Markov says OLS is **BLUE**. When they break, the residual plot tells you.

Let's simulate each violation and learn its visual signature.

In [ ]:
# Generate four synthetic residual series — one per violation type
np.random.seed(42)
n = 300
t = np.arange(n)

# A2 — Heteroskedasticity: variance changes mid-sample
het = np.random.normal(0, 1, n) * (1 + 3 * (t > 150))

# A3 — Autocorrelation: AR(1) errors
auto = np.zeros(n)
for i in range(1, n):
    auto[i] = 0.75 * auto[i-1] + np.random.normal(0, 0.5)

# Well-behaved benchmark (CLRM-compliant)
clean = np.random.normal(0, 1, n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, series, title, col in zip(
    axes,
    [clean, het, auto],
    ['CLRM-OK: random scatter', 'A2 violated: variance grows (heteroskedasticity)',
     'A3 violated: smooth runs (autocorrelation)'],
    ['black', RED, RED]
):
    ax.plot(t, series, color=GREY, lw=0.8)
    ax.axhline(0, color=col, lw=1.2)
    ax.set_title(title, fontweight='bold', fontsize=11, loc='left')
    ax.set_xlabel('Time'); ax.set_ylabel('Residual u_hat')
axes[1].axvspan(150, n, color=ORANGE, alpha=0.15)
plt.tight_layout(); plt.show()

print('• Random cloud, constant width  → assumptions probably OK')
print('• Fanning out (or in)           → heteroskedasticity (A2)')
print('• Smooth waves                  → autocorrelation (A3)')

### 7.1 Outlier impact — a single point can hijack the regression

In [ ]:
# 25 well-behaved points + 1 outlier
np.random.seed(11)
xo = np.random.uniform(0, 5, 25)
yo = 1 + 0.7 * xo + np.random.normal(0, 0.4, 25)
# Add an extreme point (e.g. a crash day)
xo_full = np.append(xo, 8)
yo_full = np.append(yo, 1.5)

# Fit with and without the outlier
m_clean, b_clean = np.polyfit(xo, yo, 1)
m_full,  b_full  = np.polyfit(xo_full, yo_full, 1)

print(f'β1_hat excluding the outlier:  {m_clean:.3f}')
print(f'β1_hat including the outlier:  {m_full:.3f}')
print(f'\n→ One extreme observation moved the slope by '
      f'{abs(m_full-m_clean)/m_clean*100:.0f}%. ')
print('  Always plot your data before trusting the slope.')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(xo, yo, color=GREY, s=50, alpha=0.8, edgecolor='black', linewidth=0.4)
ax.scatter([8], [1.5], color=RED, s=150, zorder=5, edgecolor='black', linewidth=0.8,
           label='Outlier (e.g. Black Monday)')
xx = np.linspace(0, 9, 50)
ax.plot(xx, m_clean*xx + b_clean, color='black', lw=2, label=f'Fit w/o outlier (β1_hat={m_clean:.2f})')
ax.plot(xx, m_full*xx + b_full, color=RED, lw=2, ls='--', label=f'Fit w/ outlier  (β1_hat={m_full:.2f})')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('A single outlier can hijack the regression', fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=False)
plt.tight_layout(); plt.show()

---
## Summary Table

| Concept | Key Formula | Python |
|---------|-------------|--------|
| OLS slope | $\hat{\beta}_1 = \text{Cov}(x,y) / \text{Var}(x)$ | `model.params[1]` |
| OLS intercept | $\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$ | `model.params[0]` |
| Fitted value | $\hat{Y}_t = \hat{\beta}_0 + \hat{\beta}_1 x_t$ | `model.fittedvalues` |
| Residual | $\hat{u}_t = y_t - \hat{Y}_t$ | `model.resid` |
| Property | $\sum \hat{u} = 0,\;\; \sum x \hat{u} = 0$ | mechanical |
| Add intercept | n/a | `sm.add_constant(X)` |
| Fit OLS | n/a | `sm.OLS(y, X).fit()` |

**Visual diagnostic cheat sheet:**
- Random cloud → assumptions probably OK
- Fan shape    → heteroskedasticity (A2)
- Smooth waves → autocorrelation (A3)
- One extreme point pulling the line → outlier / leverage problem

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Statistical Inference for the Simple Linear Regression Model — R², standard errors, t-tests, confidence intervals.*